In [ ]:
#Importing Essential Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#command for uploading files in notebook
from google.colab import files
uploaded = files.upload()

Saving Credit_Scoring_Dataset_500_Rows.xlsx to Credit_Scoring_Dataset_500_Rows.xlsx


In [ ]:
data = pd.read_excel('Credit_Scoring_Dataset_500_Rows.xlsx')
data.head()

,Person_Name,Age,Income,Loan_Amount,Employment,Credit_History,Behavior,Education,Marital_Status,City,Credit_Approval
0,Ananya Reddy 1,31,259206,1115586,Self-Employed,Excellent,Good,Postgraduate,Married,Chennai,0
1,Aarav Yadav 2,44,1786260,58608,Self-Employed,Poor,Good,Graduate,Single,Mumbai,0
2,Rahul Yadav 3,25,406993,739284,Government,Average,Good,Postgraduate,Married,Pune,0
3,Priya Patel 4,60,620083,166273,Salaried,Good,Good,PhD,Single,Jaipur,1
4,Isha Sharma 5,49,1437165,1385560,Freelancer,Excellent,Good,Postgraduate,Married,Kolkata,1


In [ ]:
#We will remove the unwanted columns that are of no use
data = data.drop('Person_Name', axis=1)
data.head()

In [ ]:
data.info()
data.describe()
data['Credit_Approval'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Person_Name      500 non-null    object
 1   Age              500 non-null    int64 
 2   Income           500 non-null    int64 
 3   Loan_Amount      500 non-null    int64 
 4   Employment       500 non-null    object
 5   Credit_History   500 non-null    object
 6   Behavior         500 non-null    object
 7   Education        500 non-null    object
 8   Marital_Status   500 non-null    object
 9   City             500 non-null    object
 10  Credit_Approval  500 non-null    int64 
dtypes: int64(4), object(7)
memory usage: 43.1+ KB


,count
Credit_Approval,
1,341
0,159


In [ ]:
#Converts text labels into numbers so ML models can use them.
cat_cols = ['Employment', 'Credit_History', 'Behavior', 'Education', 'Marital_Status', 'City']

le = LabelEncoder()
for col in cat_cols:
    data[col] = le.fit_transform(data[col])   # convert text categories into numbers

data.head()

In [ ]:
#Visualizes how many people were approved vs rejected — checks for class imbalance.
sns.countplot(x='Credit_Approval', data=data)
plt.title('Credit Approval Distribution (1 = Approved, 0 = Not Approved)')
plt.show()

In [ ]:
#Shows how strongly each feature relates to loan approval and to each other.
plt.figure(figsize=(10,7))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
#Creates ratio-based features that are usually stronger predictors than raw numbers alone.
data['loan_to_income'] = data['Loan_Amount'] / data['Income']       # bigger ratio = riskier applicant
data['income_per_age'] = data['Income'] / data['Age']                # rough earning-power indicator

data.head()

In [ ]:
#Splitting data for trainig
X = data.drop('Credit_Approval', axis=1)
y = data['Credit_Approval']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Logistic Regression Model
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)
log_prob = log_model.predict_proba(X_test_scaled)[:,1]

In [ ]:
#Decision Tree Model
tree_model = DecisionTreeClassifier(max_depth=6, random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
tree_prob = tree_model.predict_proba(X_test)[:,1]

In [ ]:
#Random Forest Model
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

In [ ]:
#Comparing All three Models Performance

def evaluate(name, y_true, y_pred, y_prob):
    print(f"--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred), 3))
    print("Recall   :", round(recall_score(y_true, y_pred), 3))
    print("F1-Score :", round(f1_score(y_true, y_pred), 3))
    print("ROC-AUC  :", round(roc_auc_score(y_true, y_prob), 3))
    print()

evaluate("Logistic Regression", y_test, log_pred, log_prob)
evaluate("Decision Tree", y_test, tree_pred, tree_prob)
evaluate("Random Forest", y_test, rf_pred, rf_prob)

In [ ]:
#Showing Correct vs Incorrect Decision

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Random Forest')
plt.show()

In [ ]:
#Compares how well each model distinguishes approved vs rejected across all thresholds.

for name, prob in [('Logistic Regression', log_prob), ('Decision Tree', tree_prob), ('Random Forest', rf_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, prob):.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

In [ ]:
#Feature Importance by using Random Forest Algorithm

importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importance - Random Forest')
plt.show()

## Conclusion
- Built a credit scoring model using a real-world-style dataset of 500 loan applicants.
- Features included Age, Income, Loan Amount, Employment type, Credit History, Behavior, Education, Marital Status, and City.
- Preprocessed categorical variables via Label Encoding and engineered loan-to-income and income-per-age ratios.
- Trained and compared 3 classification models: Logistic Regression, Decision Tree, Random Forest.
- Evaluated using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
- Random Forest generally performs best due to its ensemble nature, balancing precision and recall.
- Feature importance analysis highlights Credit_History, Behavior, and Loan_Amount as top predictors of credit approval.